# Multi-Label MLP with Attention + SAM

Wider backbone (1024-512-256-128) with sigmoid-gated per-bin attention and Sharpness-Aware Minimization for flatter, better-generalizing loss minima.

Compared directly against the baseline MLP from notebook 04 on the same aggregated 26,642-sample split.

Key additions: Attention gate learns which m/z bins matter for each drug. SAM optimizer finds flatter minima for better cross-species/cross-site generalization.

In [ ]:
!pip install maldideepkit maldiamrkit seaborn --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score)

from maldideepkit.attention.mlp import SpectralAttentionMLP
from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
if IN_COLAB:
    DATA_ROOT = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet/Processed")
else:
    DATA_ROOT = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet/Processed")

OUT_DIR = Path("./results_multilabel_attention")
OUT_DIR.mkdir(exist_ok=True)

CACHE_PATH = Path("../04-MultiLabel-CLassifier/aggregated_multilabel_data.npz")

BIN_COLS = [f"bin_{i}" for i in range(6000)]
DRUGS_10 = ["Ciprofloxacin", "Gentamicin", "Amoxicillin-Clavulanic acid",
            "Piperacillin-Tazobactam", "Cefepime", "Ceftriaxone",
            "Imipenem", "Ceftazidime", "Vancomycin", "Amikacin"]

print(f"Drugs: {len(DRUGS_10)}  |  Cache: {CACHE_PATH}")

In [ ]:
# =============================================================================
# 1. MULTI-LABEL DATASET
# =============================================================================

class MultiLabelDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(np.nan_to_num(Y, nan=0.0), dtype=torch.float32)
        self.mask = torch.tensor(~np.isnan(Y), dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx], self.mask[idx]

---
## Attention-Gated MLP Class (wider backbone + SAM)

In [ ]:
# =============================================================================
# 2. ATTENTION-GATED MLP WITH SAM (sklearn-compatible)
# =============================================================================

class MultiLabelMaldiMLP(BaseEstimator, ClassifierMixin):
    def __init__(self, hidden_dim=1024, head_dims=(512, 256, 128),
                 use_attention=True, dropout_high=0.3, dropout_low=0.2,
                 learning_rate=1e-3, weight_decay=0.0,
                 use_sam=True, sam_rho=0.05,
                 batch_size=64, epochs=50, early_stopping_patience=10,
                 warmup_epochs=0, val_fraction=0.1,
                 random_state=42, verbose=False, device="cpu"):
        self.hidden_dim = hidden_dim
        self.head_dims = head_dims
        self.use_attention = use_attention
        self.dropout_high = dropout_high
        self.dropout_low = dropout_low
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.use_sam = use_sam
        self.sam_rho = sam_rho
        self.batch_size = batch_size
        self.epochs = epochs
        self.early_stopping_patience = early_stopping_patience
        self.warmup_epochs = warmup_epochs
        self.val_fraction = val_fraction
        self.random_state = random_state
        self.verbose = verbose
        self.device = device

    def _build_model(self, input_dim, n_labels):
        return SpectralAttentionMLP(
            input_dim=input_dim, n_classes=n_labels,
            hidden_dim=self.hidden_dim, head_dims=self.head_dims,
            use_attention=self.use_attention,
            dropout_high=self.dropout_high, dropout_low=self.dropout_low)

    def fit(self, X, Y):
        np.random.seed(self.random_state)
        torch.manual_seed(self.random_state)

        self.n_labels_ = Y.shape[1]
        self.input_dim_ = X.shape[1]
        self.model_ = self._build_model(self.input_dim_, self.n_labels_).to(self.device)

        ds = MultiLabelDataset(X, Y)
        n_val = max(1, int(len(ds) * self.val_fraction))
        n_tr = len(ds) - n_val
        train_ds, val_ds = random_split(ds, [n_tr, n_val],
            generator=torch.Generator().manual_seed(self.random_state))
        train_loader = DataLoader(train_ds, batch_size=self.batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=self.batch_size * 2, shuffle=False)

        opt_cls = torch.optim.AdamW if self.weight_decay > 0 else torch.optim.Adam
        criterion = nn.BCEWithLogitsLoss(reduction="none")

        if self.use_sam:
            from maldideepkit.utils.sam import SAMOptimizer
            optimizer = SAMOptimizer(
                self.model_.parameters(), base_optimizer=opt_cls,
                rho=self.sam_rho, lr=self.learning_rate, weight_decay=self.weight_decay)
        else:
            optimizer = opt_cls(self.model_.parameters(), lr=self.learning_rate,
                                weight_decay=self.weight_decay)

        warmup = max(0, self.warmup_epochs)
        t_max = max(1, self.epochs - warmup)
        scheduler = (torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=t_max, eta_min=1e-6)
            if not self.use_sam else None)

        best_val_loss = float("inf"); best_state = None; patience_counter = 0

        for epoch in range(self.epochs):
            self.model_.train()
            train_loss = 0.0
            for xb, yb, mb in train_loader:
                xb, yb, mb = xb.to(self.device), yb.to(self.device), mb.to(self.device)
                if epoch < warmup and not self.use_sam:
                    for pg in optimizer.param_groups:
                        pg["lr"] = self.learning_rate * (epoch + 1) / warmup

                if self.use_sam:
                    # SAM: first forward/backward (perturbation)
                    optimizer.zero_grad()
                    logits = self.model_(xb)
                    loss = criterion(logits, yb)
                    loss = (loss * mb).sum() / mb.sum()
                    loss.backward()
                    train_loss += loss.item()
                    optimizer.first_step(zero_grad=True)
                    # SAM: second forward/backward (real update)
                    loss2 = criterion(self.model_(xb), yb)
                    loss2 = (loss2 * mb).sum() / mb.sum()
                    loss2.backward()
                    optimizer.second_step(zero_grad=True)
                else:
                    optimizer.zero_grad()
                    logits = self.model_(xb)
                    loss = criterion(logits, yb)
                    loss = (loss * mb).sum() / mb.sum()
                    loss.backward()
                    optimizer.step()
                    if epoch >= warmup: scheduler.step()
                    train_loss += loss.item()

            self.model_.eval(); val_loss = 0.0
            with torch.no_grad():
                for xb, yb, mb in val_loader:
                    xb, yb, mb = xb.to(self.device), yb.to(self.device), mb.to(self.device)
                    logits = self.model_(xb)
                    loss = criterion(logits, yb)
                    loss = (loss * mb).sum() / mb.sum()
                    val_loss += loss.item()
            val_loss /= len(val_loader); train_loss /= len(train_loader)

            if self.verbose:
                print(f"  Epoch {epoch+1:3d}/{self.epochs}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in self.model_.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= self.early_stopping_patience:
                    if self.verbose: print(f"  Early stopping at epoch {epoch+1}")
                    break

        if best_state is not None:
            self.model_.load_state_dict(best_state)
        self.model_.eval()
        self.is_fitted_ = True
        return self

    def predict_proba(self, X):
        X_t = torch.tensor(X, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            return torch.sigmoid(self.model_(X_t)).cpu().numpy()

    def predict(self, X, thresholds=None):
        proba = self.predict_proba(X)
        if thresholds is None: thresholds = [0.5] * self.n_labels_
        return (proba >= np.array(thresholds)).astype(int)

    def get_attention_weights(self, X):
        self.model_.eval()
        X_t = torch.tensor(X, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            _ = self.model_(X_t)
        if self.model_.last_attention is None:
            raise RuntimeError("Attention not captured -- set use_attention=True")
        return self.model_.last_attention.cpu().numpy()

---
## Load Cached Data + Split

In [ ]:
# =============================================================================
# 3. LOAD DATA + SPLIT (70/15/15) - FROM 04 CACHE
# =============================================================================

if CACHE_PATH.exists():
    print(f"Loading aggregated data from cache: {CACHE_PATH}")
    cached = np.load(CACHE_PATH, allow_pickle=True)
    X_all = cached['X_all']; Y_all = cached['Y_all']; sp_all = cached['sp_all']
else:
    raise FileNotFoundError(f"Cache not found: {CACHE_PATH}. Run 04 first.")

print(f"Total: {X_all.shape[0]} samples, {len(np.unique(sp_all))} species")

idx_trval, idx_test, _, _ = stratified_species_drug_split(
    np.arange(len(Y_all)).reshape(-1, 1), np.zeros(len(Y_all)),
    species=sp_all, test_size=0.15, random_state=SEED)
idx_trval = idx_trval.flatten().astype(int); idx_test = idx_test.flatten().astype(int)
X_trval, Y_trval, sp_trval = X_all[idx_trval], Y_all[idx_trval], sp_all[idx_trval]
X_test, Y_test = X_all[idx_test], Y_all[idx_test]

val_frac = 0.15 / 0.85
idx_train, idx_val, _, _ = stratified_species_drug_split(
    np.arange(len(Y_trval)).reshape(-1, 1), np.zeros(len(Y_trval)),
    species=sp_trval, test_size=val_frac, random_state=SEED)
idx_train = idx_train.flatten().astype(int); idx_val = idx_val.flatten().astype(int)
X_train, Y_train = X_trval[idx_train], Y_trval[idx_train]
X_val, Y_val = X_trval[idx_val], Y_trval[idx_val]
print(f"Split: train={X_train.shape[0]}  val={X_val.shape[0]}  test={X_test.shape[0]}")

state = fit_input_transform(X_train, "log1p+standardize")
X_train_pp = apply_input_transform(X_train, state); X_val_pp = apply_input_transform(X_val, state)
X_test_pp = apply_input_transform(X_test, state)
print("Preprocessing done.")

---
## Grid Search (SAM + Attention, 6x6 lr x dropout)

In [ ]:
# =============================================================================
# 4. GRID SEARCH: lr x dropout (SAM + Attention)
# =============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

LR_GRID = np.linspace(5e-5, 5e-4, 6)
DROP_GRID = np.linspace(0.1, 0.5, 6)
thresholds = np.linspace(0.05, 0.95, 91)
print(f"Grid: {len(LR_GRID)} lr x {len(DROP_GRID)} dropout = {len(LR_GRID)*len(DROP_GRID)} combos")

best_balacc = -1; best_lr = None; best_dh = None; best_dl = None

for lr in LR_GRID:
    for d in DROP_GRID:
        dh, dl = d, d / 2
        mlp = MultiLabelMaldiMLP(
            hidden_dim=1024, head_dims=(512, 256, 128),
            use_attention=True, dropout_high=dh, dropout_low=dl,
            learning_rate=lr, weight_decay=1e-3,
            use_sam=True, sam_rho=0.05,
            batch_size=64, epochs=50, early_stopping_patience=10,
            warmup_epochs=10, val_fraction=0.1,
            random_state=SEED, verbose=False, device=device)
        mlp.fit(X_train_pp, Y_train)

        proba_v = mlp.predict_proba(X_val_pp)
        per_drug_ba = []
        for di in range(10):
            vm = ~np.isnan(Y_val[:, di])
            if vm.sum() < 2: continue
            pv = proba_v[vm, di]; yv = Y_val[vm, di].astype(int)
            bt = thresholds[np.argmax([balanced_accuracy_score(yv, pv >= t) for t in thresholds])]
            per_drug_ba.append(balanced_accuracy_score(yv, pv >= bt))
        macro_ba = np.mean(per_drug_ba) if per_drug_ba else 0.0
        mark = " *" if macro_ba > best_balacc else ""
        print(f"  lr={lr:.1e}  drop=({dh:.2f},{dl:.2f})  macro_BalAcc={macro_ba:.4f}{mark}")
        if macro_ba > best_balacc:
            best_balacc = macro_ba; best_lr = lr; best_dh = dh; best_dl = dl

print(f"\nBest: lr={best_lr:.1e} drop=({best_dh:.2f},{best_dl:.2f})  macro_BalAcc={best_balacc:.4f}")

---
## Best Model Retrain + Threshold Tuning

In [ ]:
# =============================================================================
# 5. RETRAIN BEST + THRESHOLD TUNING
# =============================================================================

mlp_attn = MultiLabelMaldiMLP(
    hidden_dim=1024, head_dims=(512, 256, 128),
    use_attention=True, dropout_high=best_dh, dropout_low=best_dl,
    learning_rate=best_lr, weight_decay=1e-4,
    use_sam=True, sam_rho=0.05,
    batch_size=64, epochs=100, early_stopping_patience=15,
    warmup_epochs=10, val_fraction=0.1,
    random_state=SEED, verbose=True, device=device)
mlp_attn.fit(X_train_pp, Y_train)

proba_val = mlp_attn.predict_proba(X_val_pp)
attn_thresholds = []
for di in range(10):
    vm = ~np.isnan(Y_val[:, di])
    if vm.sum() < 2: attn_thresholds.append(0.5); continue
    pv = proba_val[vm, di]; yv = Y_val[vm, di].astype(int)
    bt = thresholds[np.argmax([balanced_accuracy_score(yv, pv >= t) for t in thresholds])]
    attn_thresholds.append(bt)
attn_thresholds = np.array(attn_thresholds)
print(f"\nPer-drug thresholds: {[f'{t:.2f}' for t in attn_thresholds]}")

proba_test = mlp_attn.predict_proba(X_test_pp)
attn_results = {}
for di, drug in enumerate(DRUGS_10):
    tm = ~np.isnan(Y_test[:, di])
    if tm.sum() < 2: attn_results[drug] = np.nan; continue
    preds = (proba_test[tm, di] >= attn_thresholds[di])
    yt = Y_test[tm, di].astype(int)
    attn_results[drug] = balanced_accuracy_score(yt, preds)

print("\nAttention+SAM MLP -- Test BalAcc:")
for drug, ba in attn_results.items():
    print(f"  {drug:35s}  {ba:.4f}")

---
## Attention Visualization

In [ ]:
# =============================================================================
# 6. ATTENTION VISUALIZATION
# =============================================================================

attn_weights = mlp_attn.get_attention_weights(X_test_pp)
mz_proj = np.arange(attn_weights.shape[1])
mean_attn = attn_weights.mean(axis=0)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].bar(mz_proj, mean_attn, width=1, color="steelblue", alpha=0.7)
axes[0].set_xlabel("Projected bin index (attention gate, 1024-dim)")
axes[0].set_ylabel("Mean attention weight")
axes[0].set_title("Attention Gate Profile (avg across all test samples)")
axes[0].axhline(y=0.5, color="red", ls="--", alpha=0.4, label="neutral gate")
axes[0].legend()

top_n = 20
top_idx = np.argsort(mean_attn)[-top_n:]
axes[1].barh(range(top_n), mean_attn[top_idx][::-1], color="orange", alpha=0.7)
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels([f"bin {i}" for i in top_idx[::-1]], fontsize=8)
axes[1].set_xlabel("Attention weight")
axes[1].set_title(f"Top {top_n} Most Attended Projected Bins")
axes[1].axvline(x=0.5, color="red", ls="--", alpha=0.4)

plt.tight_layout()
plt.savefig(OUT_DIR / "attention_profile.pdf", bbox_inches="tight")
plt.show()

print("\nTop 5 attended bins:", top_idx[::-1][:5].tolist())

---
## Baseline MLP vs Attention+SAM MLP

In [ ]:
# =============================================================================
# 7. COMPARISON: Baseline MLP vs Attention+SAM
# =============================================================================

_baseline = {
    "Ciprofloxacin": 0.7662, "Gentamicin": 0.8332,
    "Amoxicillin-Clavulanic acid": np.nan, "Piperacillin-Tazobactam": 0.8179,
    "Cefepime": 0.8683, "Ceftriaxone": 0.8091,
    "Imipenem": 0.8949, "Ceftazidime": 0.7849,
    "Vancomycin": 0.8615, "Amikacin": 0.7056,
}

comp_rows = []
for drug in DRUGS_10:
    comp_rows.append({
        "Drug": drug[:15],
        "Baseline_MLP": _baseline.get(drug, np.nan),
        "Attention+SAM": attn_results.get(drug, np.nan),
    })
df_comp = pd.DataFrame(comp_rows)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(df_comp)); w = 0.35
ax.bar(x - w/2, df_comp["Baseline_MLP"], w, label="Baseline MLP (04)", color="#aec7e8")
ax.bar(x + w/2, df_comp["Attention+SAM"], w, label="Attention+SAM (04-02)", color="#1f77b4")
ax.set_xticks(x); ax.set_xticklabels(df_comp["Drug"], fontsize=8, rotation=45, ha="right")
ax.set_ylabel("Test BalAcc"); ax.set_title("Baseline MLP vs Attention+SAM MLP")
ax.legend(); ax.set_ylim(0, 1); ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
plt.grid(True, ls='--', lw=0.5, color='gray', alpha=0.7)
plt.tight_layout(); plt.savefig(OUT_DIR / "attn_vs_baseline.pdf")
plt.show()

print("\nBaseline vs Attention+SAM:")
print(df_comp.to_string(index=False))

In [ ]:
print("\nDone. Attention+SAM results saved to", OUT_DIR.resolve())
import joblib
save_dir = OUT_DIR / "trained_model"
save_dir.mkdir(exist_ok=True)
joblib.dump({"state": state, "thresholds": attn_thresholds,
             "best_lr": best_lr, "best_dh": best_dh, "best_dl": best_dl,
             "results": attn_results}, save_dir / "metadata.joblib")
torch.save(mlp_attn.model_.state_dict(), save_dir / "model.pt")
print(f"Model saved to {save_dir}")